In [1]:
import os
import argparse
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import h5py
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from typing import List, Set, Optional


### 数据预处理 

In [ ]:
import numpy as np
import pickle
from scipy import sparse
import numpy as np
import qnorm
from scipy.sparse import csr_matrix
from collections import defaultdict
import pandas as pd
import gzip
import argparse
import os
import scanpy as sc
import json
import gc
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import euclidean_distances

In [2]:
adata_h5 = sc.read_h5ad("./data/MERFISH/Seg/H5/20/moffitt_MERFISH_20_PASTEaligned.h5ad")
print(adata_h5)
gene_count_before = len(list(adata_h5.var_names))
sc.pp.filter_genes(adata_h5, min_cells = 1)
gene_count_after = len(list(adata_h5.var_names))
print('Gene filtering done. Number of genes reduced from %d to %d' %(gene_count_before, gene_count_after))

AnnData object with n_obs × n_vars = 21993 × 161
    obs: 'Cell_ID', 'Animal_ID', 'Animal_sex', 'Behavior', 'Bregma', 'Centroid_X', 'Centroid_Y', 'Cell_class', 'Neuron_cluster_ID'
    var: 'n_cells'
    obsm: 'spatial', 'spatial_3d_aligned', 'spatial_aligned', 'spatial_raw'
Gene filtering done. Number of genes reduced from 161 to 161


In [3]:
gene_ids = list(adata_h5.var_names)
print(len(gene_ids))
gene_ids = [gene_id for gene_id in gene_ids]
coordinates_xy = np.array(adata_h5.obsm['spatial'])
bregma_values = adata_h5.obs["Bregma"].astype(float).values
bregma_z = adata_h5.obs["Bregma"].astype(float).values.reshape(-1, 1)  * 1000
coordinates = np.concatenate([coordinates_xy[:, :2], bregma_z], axis=1)



161


In [4]:
a = len(coordinates[:,0])#提取所有的x坐标
print(coordinates.shape)
coordinates[:][1]
type(np.array(adata_h5.obs_names))
cell_barcode = np.array(adata_h5.obs_names)#6605*1
print('Number of barcodes: %d'%cell_barcode.shape[0])

(21993, 3)
Number of barcodes: 21993


In [6]:
temp = qnorm.quantile_normalize(np.transpose(sparse.csr_matrix.toarray(adata_h5.X)))  #https://en.wikipedia.org/wiki/Quantile_normalization
cell_vs_gene = np.transpose(temp)

In [ ]:

i= 0 
barcode_info = []
for cell_code in cell_barcode:
    barcode_info.append([cell_code, coordinates[i, 0], coordinates[i, 1], 0, coordinates[i, 2]])
    i = i+1
print(type(barcode_info))


gene_info = dict()
for gene in gene_ids:
    gene_info[gene] = ""
print(type(gene_info))


gene_index = dict()
i = 0
for gene in gene_ids:
    gene_index[gene] = i
    i = i+1

<class 'list'>
<class 'dict'>


In [ ]:

k=10
print('Neighborhood distance is set to be %d nearest neighbors' %k)
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='kd_tree', n_jobs=-1)
nbrs.fit(coordinates)
distances, indices = nbrs.kneighbors(coordinates)#indices是十个最近邻的细胞id，cell*cellid
distances = distances[:, 1:]
indices = indices[:, 1:]


Neighborhood distance is set to be 10 nearest neighbors


In [ ]:

distance_a_b = np.median(distances[:, 0])#去掉自身细胞了，distances[:, 0]距离当前细胞最近的距离典型最近邻距离
distance_a_b

12.935496014101071

In [10]:
print('Assign weight to the neighborhood relations based on neighborhood distance')
weightdict_i_to_j = defaultdict(dict)
for cell_idx in range (0, indices.shape[0]):
    max_value = np.max(distances[cell_idx,:])
    min_value = np.min(distances[cell_idx,:])
    for neigh_idx in range (0, indices.shape[1]):
        neigh_cell_idx = indices[cell_idx][neigh_idx]
        distance_neigh_cell = distances[cell_idx][neigh_idx]
        flipped_distance_neigh_cell = 1-(distance_neigh_cell-min_value)/(max_value-min_value)
        weightdict_i_to_j[neigh_cell_idx][cell_idx] = flipped_distance_neigh_cell


Assign weight to the neighborhood relations based on neighborhood distance


In [ ]:

juxtacrine_distance = distance_a_b 
print("Auto calculated juxtacrine distance is %g. "%juxtacrine_distance)

Auto calculated juxtacrine distance is 12.9355. 


In [ ]:

i=0
node_id_sorted_xy=[]
for cell_code in cell_barcode:
    node_id_sorted_xy.append([i, coordinates[i,0],coordinates[i,1]])
    i = i+1
node_id_sorted_xy = sorted(node_id_sorted_xy, key = lambda x: (x[1], x[2]))#优先比较x
node_id_sorted_xy[0]

[11128, -1240.0837231244432, 2.185018925327313]

In [ ]:

print('ligand-receptor database reading.')
df = pd.read_csv("./data/LR/Cell_mouse_database.csv", sep=",")
# df["Ligand"] = df["Ligand"].astype(str).str.upper()
# df["Receptor"] = df["Receptor"].astype(str).str.upper()
print('ligand-receptor database reading done.')
print('Preprocess start.')
ligand_dict_dataset = defaultdict(list)
cell_cell_contact = dict() 
count_pair = 0
for i in range (0, df["Ligand"].shape[0]):
    ligand = df["Ligand"][i]
    if ligand not in gene_info: # not found in the dataset
        continue 
    receptor = df["Receptor"][i]
    if receptor not in gene_info: # not found in the dataset
        continue
        
    ligand_dict_dataset[ligand].append(receptor) 
    gene_info[ligand] = 'included'
    gene_info[receptor] = 'included'
    count_pair = count_pair + 1
    if df["Annotation"][i] == 'Cell-Cell Contact':
        cell_cell_contact[(ligand, receptor)] = ''
for ligand in list(ligand_dict_dataset.keys()):
    if not ligand_dict_dataset[ligand]:
        del ligand_dict_dataset[ligand]
print('number of ligands %d '%len(ligand_dict_dataset.keys()))

included_gene=[]
for gene in gene_info.keys():
    if gene_info[gene] == 'included':
        included_gene.append(gene)
print('Total genes in this dataset: %d, number of genes working as ligand and receptor: %d，LR-pair: %d '%(len(gene_ids),len(included_gene),count_pair))
    

ligand-receptor database reading.
ligand-receptor database reading done.
Preprocess start.
number of ligands 20 
Total genes in this dataset: 161, number of genes working as ligand and receptor: 50，LR-pair: 142 


In [ ]:

l_r_pair = dict()
lr_id = 0
for gene in list(ligand_dict_dataset.keys()): #20个配体
    ligand_dict_dataset[gene]=list(set(ligand_dict_dataset[gene]))
    receptors = list(set(ligand_dict_dataset[gene]))
    if not receptors: # 如果列表为空，跳过该配体，不为其创建条目
        continue
    l_r_pair[gene] = dict()
    for receptor_gene in ligand_dict_dataset[gene]:
        l_r_pair[gene][receptor_gene] = lr_id 
        lr_id  = lr_id  + 1
print('number of ligand-receptor pairs in this dataset %d '%lr_id)


number of ligand-receptor pairs in this dataset 142 


In [ ]:

cell_percentile = []
cell_active_count = []
for i in range (0, cell_vs_gene.shape[0]):
    y = sorted(cell_vs_gene[i]) 
    active_cutoff = np.percentile(y, 90)
    if active_cutoff == min(cell_vs_gene[i][:]):
        times = 1
        while active_cutoff == min(cell_vs_gene[i][:]):
            new_threshold = 80 + 5 * times
            if new_threshold >= 100:
                active_cutoff = max(cell_vs_gene[i][:])  
                if active_cutoff == min(cell_vs_gene[i][:]):
                    active_cutoff = max(cell_vs_gene[i][:]) + 1 
                break
            active_cutoff = np.percentile(y, new_threshold)
            times = times + 1
    cell_percentile.append(active_cutoff)
    active_count = np.sum(cell_vs_gene[i] >= active_cutoff)
    cell_active_count.append(active_count)
    
print('set threshold gene percentile done.')
        
        

set threshold gene percentile done.


In [ ]:

count_total_edges = 0
cells_ligand_vs_receptor = defaultdict(dict)

ligand_list =  list(ligand_dict_dataset.keys())            
start_index = 0 
end_index = len(ligand_list) 
print('some preprocessing before making the input graph')    
for g in range(start_index, end_index):  
    gene = ligand_list[g]#配体
    for i in weightdict_i_to_j:
        if cell_vs_gene[i][gene_index[gene]] < cell_percentile[i]: 
            continue            
        for j in weightdict_i_to_j[i]: 
            for gene_rec in ligand_dict_dataset[gene]:
                if cell_vs_gene[j][gene_index[gene_rec]] >= cell_percentile[j]: 
                    communication_score = cell_vs_gene[i][gene_index[gene]] * cell_vs_gene[j][gene_index[gene_rec]]
                    relation_id = l_r_pair[gene][gene_rec]

                    if communication_score<=0:
                        print('zero valued ccc score found. Might be a potential ERROR!! ')
                        continue
                    if i in cells_ligand_vs_receptor:
                        if j in cells_ligand_vs_receptor[i]:
                            cells_ligand_vs_receptor[i][j].append([gene, gene_rec, communication_score, relation_id])
                        else:
                            cells_ligand_vs_receptor[i][j] = []
                            cells_ligand_vs_receptor[i][j].append([gene, gene_rec, communication_score, relation_id])
                    else:
                        cells_ligand_vs_receptor[i][j] = []
                        cells_ligand_vs_receptor[i][j].append([gene, gene_rec, communication_score, relation_id])

                    count_total_edges = count_total_edges + 1

    print('%d/%d ligand genes processed'%(g+1, len(ligand_list)), end='\r')

print('') 
print('total number of edges in the input graph %d '%count_total_edges)

some preprocessing before making the input graph
20/20 ligand genes processed
total number of edges in the input graph 152614 


In [ ]:
# input graph generation
ccc_index_dict = dict()
row_col = [] 
edge_weight = [] 
lig_rec = [] 
self_loop_found = defaultdict(dict) 
for i in cells_ligand_vs_receptor.keys():
    for j in cells_ligand_vs_receptor[i].keys():
        if i in weightdict_i_to_j and j in weightdict_i_to_j[i]: 
            count_local = 0 
            if len(cells_ligand_vs_receptor[i][j])>0:
                for k in range (0, len(cells_ligand_vs_receptor[i][j])):
                    gene = cells_ligand_vs_receptor[i][j][k][0] #作用配体
                    gene_rec = cells_ligand_vs_receptor[i][j][k][1] #作用受体
                    ligand_receptor_coexpression_score = cells_ligand_vs_receptor[i][j][k][2]
                    row_col.append([i,j])
                    edge_weight.append([weightdict_i_to_j[i][j], ligand_receptor_coexpression_score, cells_ligand_vs_receptor[i][j][k][3]])
                    lig_rec.append([gene, gene_rec])

                    if i==j: 
                        self_loop_found[i][j] = ''


total_num_cell = cell_vs_gene.shape[0]
print('total number of nodes is %d, and edges is %d in the input graph'%(total_num_cell, len(row_col)))
print('preprocess done.')
print('writing data ...')


total number of nodes is 21993, and edges is 152614 in the input graph
preprocess done.
writing data ...


In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument( '--data_name', type=str, default='moffitt_MERFISH_20_aligned', help='Name of the dataset')  
parser.add_argument( '--data_to', type=str, default='./outp/input_graph/', help='Path to save the input graph (to be passed to GAT)')
parser.add_argument( '--metadata_to', type=str, default='./outp/metadata/', help='Path to save the metadata')
parser.add_argument('-f', type=str, help='Jupyter kernel connection file (ignored)')
args = parser.parse_args()


if args.data_to == './outp/input_graph/':
    args.data_to = args.data_to + args.data_name + '/'
if not os.path.exists(args.data_to):
    os.makedirs(args.data_to)

if args.metadata_to == './outp/metadata/':
    args.metadata_to = args.metadata_to + args.data_name + '/'
if not os.path.exists(args.metadata_to):
    os.makedirs(args.metadata_to)


with gzip.open(args.data_to + args.data_name + '_adjacency_records', 'wb') as fp:  
    pickle.dump([row_col, edge_weight, lig_rec, total_num_cell], fp)

with gzip.open(args.metadata_to + args.data_name +'_self_loop_record', 'wb') as fp: 
    pickle.dump(self_loop_found, fp)

with gzip.open(args.metadata_to + args.data_name +'_barcode_info', 'wb') as fp:  
    pickle.dump(barcode_info, fp)

df = pd.DataFrame(gene_ids)
df.to_csv(args.metadata_to + 'gene_ids_'+args.data_name+'.csv', index=False, header=False)
df = pd.DataFrame(cell_barcode)
df.to_csv(args.metadata_to + 'cell_barcode_'+args.data_name+'.csv', index=False, header=False)
df = pd.DataFrame(coordinates)
df.to_csv(args.metadata_to + 'coordinates_'+args.data_name+'.csv', index=False, header=False)


In [ ]:

with gzip.open(args.data_to + args.data_name + '_cell_vs_gene_quantile_transformed', 'wb') as fp:  
    pickle.dump(cell_vs_gene, fp)

cell_annotation = adata_h5.obs[["Cell_ID", "Cell_class"]].copy()
cell_annotation = cell_annotation.rename(columns={
    "Cell_ID": "Barcode",
    "Cell_class":"Type"
})

cell_vs_annotation = dict()
for i in range(0, adata_h5.n_obs):
    cell_vs_annotation[cell_annotation['Barcode'].iloc[i]] = cell_annotation['Type'].iloc[i]

unique_annots = np.unique(list(cell_annotation['Type']))
unique_annots_count = len(unique_annots)
print('unique type found %d'%unique_annots_count)
type_vs_id = dict()
for i in range(0, len(unique_annots)):
    type_vs_id[unique_annots[i]] = i


node_features = np.zeros((len(cell_barcode), unique_annots_count))
for i in range(0, len(cell_barcode)):
    node_features[i][type_vs_id[cell_vs_annotation[cell_barcode[i]]]] = 1

with gzip.open(args.data_to + args.data_name + '_cell_vs_feature', 'wb') as fp:  
    pickle.dump(node_features, fp)


unique type found 16


In [20]:
f = gzip.open(args.data_to + args.data_name + '_adjacency_records', 'rb')
row_col, edge_weight, lig_rec, num_cell = pickle.load(f)

###  Pseudo-positive

In [23]:
from collections import defaultdict
from scipy.stats import entropy
from sklearn.preprocessing import minmax_scale


In [24]:
def robust_quantile_threshold(arr, q, default=None):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return default
    return float(np.quantile(arr, q))

In [25]:
gene_ids = adata_h5.var_names.tolist() 
obs_names = adata_h5.obs['Cell_ID'].astype(str).tolist()
cell_class = adata_h5.obs['Cell_class'].astype(str).tolist()
cell_classes = np.array(cell_class)
coordinates  = adata_h5.obsm['spatial'].astype(float)

group_names = pd.unique(cell_classes).tolist()
group_to_idx = {g: i for i, g in enumerate(group_names)}
group_ids = np.array([group_to_idx[g] for g in cell_classes], dtype=int)
n_groups = len(group_names)

print("Using Cell_class as groups. #groups =", n_groups)

Using Cell_class as groups. #groups = 16


In [ ]:
lr_df = pd.read_csv("./data/LR/Cell_mouse_database.csv", sep=",")
lr_df = lr_df.copy()
lr_df["Ligand"] = lr_df["Ligand"].astype(str)
lr_df["Receptor"] = lr_df["Receptor"].astype(str)

keep = lr_df["Ligand"].isin(gene_index) & lr_df["Receptor"].isin(gene_index)
lr_df = lr_df.loc[keep].reset_index(drop=True)
print("LR pairs kept in 160-gene panel:", len(lr_df))


ligand_to_receptors = defaultdict(list)
receptor_to_ligands = defaultdict(list)

for _, row in lr_df.iterrows():
    lig = row["Ligand"]
    rec = row["Receptor"]
    ligand_to_receptors[lig].append(rec)
    receptor_to_ligands[rec].append(lig)

cell_cell_contact_receptors = set(
    lr_df.loc[lr_df["Annotation"].astype(str) == "Cell-Cell Contact", "Receptor"].astype(str).tolist()
)

LR pairs kept in 160-gene panel: 142


In [27]:
tf_target_file = "./database/mouse_tf_target.csv" 
tf_df = pd.read_csv(tf_target_file)
tf_targets = defaultdict(list)
for i,row in tf_df.iterrows():
    tf_targets[row['source']].append(row['target'])


In [ ]:
gene_index = {g: i for i, g in enumerate(gene_ids)}

group_names = pd.unique(cell_classes).tolist()
group_to_idx = {g: i for i, g in enumerate(group_names)}
group_ids = np.array([group_to_idx[g] for g in cell_classes], dtype=int)
n_groups = len(group_names)

print("Using Cell_class as groups. #groups =", n_groups)

lr_df = lr_df.copy()
lr_df["Ligand"] = lr_df["Ligand"].astype(str)
lr_df["Receptor"] = lr_df["Receptor"].astype(str)

keep = lr_df["Ligand"].isin(gene_index) & lr_df["Receptor"].isin(gene_index)
lr_df = lr_df.loc[keep].reset_index(drop=True)
print("LR pairs kept in 160-gene panel:", len(lr_df))

ligand_to_receptors = defaultdict(list)
receptor_to_ligands = defaultdict(list)

for _, row in lr_df.iterrows():
    lig = row["Ligand"]
    rec = row["Receptor"]
    ligand_to_receptors[lig].append(rec)
    receptor_to_ligands[rec].append(lig)

cell_cell_contact_receptors = set(
    lr_df.loc[lr_df["Annotation"].astype(str) == "Cell-Cell Contact", "Receptor"].astype(str).tolist()
)

print("Computing group mean expression ...")
meanexpr = np.zeros((len(gene_ids), n_groups), dtype=float)

for g in range(n_groups):
    idx = np.where(group_ids == g)[0]
    if len(idx) == 0:
        continue
    elif len(idx) >= 2:
        meanexpr[:, g] = cell_vs_gene[idx].mean(axis=0)
    else:
        meanexpr[:, g] = cell_vs_gene[idx[0]]
        
def entropy(vec):
    vec = vec + 1e-8
    p = vec / np.sum(vec)
    return -np.sum(p * np.log(p))

def sigmoid(x, a=10):
    return 1 / (1 + np.exp(-a*x))


Using Cell_class as groups. #groups = 16
LR pairs kept in 160-gene panel: 142
Computing group mean expression ...


In [ ]:
print("Computing w1 ...")
w1_dict = {}
for lig, rec in lig_rec:
    lig_idx = gene_index.get(lig)
    rec_idx = gene_index.get(rec)
    if lig_idx is None or rec_idx is None:
        w1_dict[(lig, rec)] = 0.0
        continue
    lig_expr = meanexpr[lig_idx, :] + 1e-3 
    rec_expr = meanexpr[rec_idx, :] + 1e-3  
    ent_l = entropy(lig_expr)
    ent_r = entropy(rec_expr)
    w1_dict[(lig, rec)] = 1.0 / (1 + ent_l + ent_r)


vals = np.array(list(w1_dict.values()))
median_val = np.median(vals[vals>0]) if np.any(vals>0) else 1.0
for k in w1_dict.keys():
    w1_dict[k] = float(w1_dict[k]/median_val*0.5)

print("Computing w2 ...")
eta = 0.1
for lig, rec in lig_rec:
    ligand_to_receptors[lig].append(rec)
    receptor_to_ligands[rec].append(lig)

w2_dict = {}
for lig, rec in lig_rec:
    lig_idx = gene_index.get(lig)
    rec_idx = gene_index.get(rec)
    if lig_idx is None or rec_idx is None:
        w2_dict[(lig, rec)] = np.zeros((n_groups, n_groups))
        continue

    competing_ligs = receptor_to_ligands[rec]
    competing_recs = ligand_to_receptors[lig]

    a_l_by_group = np.zeros(n_groups)
    for gs in range(n_groups):
        lig_expr = meanexpr[lig_idx, gs]
        comp_exprs = [meanexpr[gene_index[x], gs] for x in competing_ligs if x in gene_index]
        max_l = np.max(comp_exprs) if comp_exprs else lig_expr
        num = np.exp(eta*lig_expr)-1
        den = np.exp(eta*max_l)-1
        a_l_by_group[gs] = 0 if abs(den)<1e-12 else num/den

    a_r_by_group = np.zeros(n_groups)
    for gr in range(n_groups):
        rec_expr = meanexpr[rec_idx, gr]
        comp_exprs = [meanexpr[gene_index[x], gr] for x in competing_recs if x in gene_index]
        max_r = np.max(comp_exprs) if comp_exprs else rec_expr
        num = np.exp(eta*rec_expr)-1
        den = np.exp(eta*max_r)-1
        a_r_by_group[gr] = 0 if abs(den)<1e-12 else num/den

    mat = np.zeros((n_groups, n_groups))
    for gs in range(n_groups):
        for gr in range(n_groups):
            mat[gs, gr] = np.sqrt((a_l_by_group[gs]**2 + a_r_by_group[gr]**2)/2)
    w2_dict[(lig, rec)] = mat
    
    
print("Computing w3 ...")

tf_edge_list = []
for tf, targets in tf_targets.items():
    for tg in targets:
        if tf in gene_index and tg in gene_index:
            tf_edge_list.append((str(tf), str(tg)))

tfcor = np.zeros((len(tf_edge_list), n_groups), dtype=float)

for l, (tf_gene, target_gene) in enumerate(tf_edge_list):
    tf_vec_all = cell_vs_gene[:, gene_index[tf_gene]]
    tg_vec_all = cell_vs_gene[:, gene_index[target_gene]]

    for k in range(n_groups):
        idx = np.where(group_ids == k)[0]
        if len(idx) < 2:
            tfcor[l, k] = 0.0
            continue

        x = tf_vec_all[idx].astype(float)
        y = tg_vec_all[idx].astype(float)

        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            tfcor[l, k] = 0.0
        else:
            tfcor[l, k] = np.corrcoef(x, y)[0, 1]

tfcor[np.isnan(tfcor)] = 0.0

tf_targets_arr = np.array([x[1] for x in tf_edge_list], dtype=object)

w3_dict = {}

for lig, rec in lig_rec:
    if lig not in gene_index or rec not in gene_index:
        w3_dict[(lig, rec)] = np.ones((n_groups, n_groups), dtype=float) * sigmoid(0.0, a=10)
        continue

    lcor = np.zeros(n_groups, dtype=float)
    rcor = np.zeros(n_groups, dtype=float)

    lig_mask = np.where(tf_targets_arr == lig)[0]
    rec_mask = np.where(tf_targets_arr == rec)[0]

    for k in range(n_groups):
        if len(lig_mask) > 0:
            lcor[k] = abs(np.mean(tfcor[lig_mask, k]))
        else:
            lcor[k] = 0.0

        if len(rec_mask) > 0:
            rcor[k] = abs(np.mean(tfcor[rec_mask, k]))
        else:
            rcor[k] = 0.0

    lcor[np.isnan(lcor)] = 0.0
    rcor[np.isnan(rcor)] = 0.0

    mat = (
        np.repeat(lcor[:, None], n_groups, axis=1) +
        np.repeat(rcor[None, :], n_groups, axis=0)
    ) / 2.0

    mat = sigmoid(mat, a=10)
    w3_dict[(lig, rec)] = mat
print("Computing scComm-like edge scores ...")

lr_coexp_all = np.array([float(x[1]) for x in edge_weight], dtype=float)
lr_coexp_norm = minmax_scale(np.log1p(lr_coexp_all))

edge_meta_rows = []
score_raw = []
w1_list = []
w2_list = []
w3_list = []

for k in range(len(row_col)):
    src, dst = row_col[k]
    src = int(src)
    dst = int(dst)

    distance_weight = float(edge_weight[k][0])
    lr_coexp_score = float(edge_weight[k][1])
    relation_id = int(edge_weight[k][2])

    lig = str(lig_rec[k][0])
    rec = str(lig_rec[k][1])

    if lig not in gene_index or rec not in gene_index:
        continue

    gs = group_ids[src]
    gr = group_ids[dst]

    w1 = w1_dict.get((lig, rec), 0.0)
    w2 = w2_dict.get((lig, rec), np.zeros((n_groups, n_groups), dtype=float))[gs, gr]

    w3 = w3_dict.get((lig, rec), np.ones((n_groups, n_groups), dtype=float) * sigmoid(0.0, a=10))[gs, gr]

    w1_list.append(w1)
    w2_list.append(w2)
    w3_list.append(w3)

    base_score = lr_coexp_norm[k]
    final_score = float(base_score * w1 * w2 * w3)

    pair_distance = float(np.linalg.norm(coordinates[src] - coordinates[dst]))
    is_contact = 1 if rec in cell_cell_contact_receptors else 0

    score_raw.append(final_score)
    edge_meta_rows.append({
        "edge_idx": k,
        "src": src,
        "dst": dst,
        "ligand": lig,
        "receptor": rec,
        "group_src": group_names[gs],
        "group_dst": group_names[gr],
        "distance_weight": distance_weight,
        "lr_coexp_score": lr_coexp_score,
        "lr_coexp_norm": float(base_score),
        "relation_id": relation_id,
        "w1": float(w1),
        "w2": float(w2),
        "w3": float(w3),
        "pair_distance": pair_distance,
        "is_contact": is_contact,
        "raw_score": final_score,
    })

edge_meta_df = pd.DataFrame(edge_meta_rows)
edge_meta_df.to_csv(args.data_to + args.data_name +'_edge_meta_df.csv')
if len(edge_meta_df) == 0:
    raise RuntimeError("No valid candidate edges after filtering.")

edge_meta_df["sccomm_like_score"] = minmax_scale(edge_meta_df["raw_score"].values.astype(float))
print("edge_meta_df save success...")

Computing w1 ...
Computing w2 ...


In [ ]:
print("Generating pseudo labels ...")

score_arr = edge_meta_df["sccomm_like_score"].values
dw_arr = edge_meta_df["distance_weight"].values
dist_arr = edge_meta_df["pair_distance"].values
# 更偏高置信
q95_score = robust_quantile_threshold(score_arr, 0.90, 1.0)
q10_score = robust_quantile_threshold(score_arr, 0.15, 0.0)
q85_dw = robust_quantile_threshold(dw_arr, 0.80, 1.0)
q25_dw = robust_quantile_threshold(dw_arr, 0.15, 0.0)


pseudo_labels = np.full(len(edge_meta_df), -1, dtype=np.int64)

for i in range(len(edge_meta_df)):
    score = float(edge_meta_df.iloc[i]["sccomm_like_score"])
    dw = float(edge_meta_df.iloc[i]["distance_weight"])
    dist = float(edge_meta_df.iloc[i]["pair_distance"])
    is_contact = int(edge_meta_df.iloc[i]["is_contact"])

    is_pos = (score >= q95_score) and (dw >= q85_dw)
    if is_pos and is_contact == 1 and dist > juxtacrine_distance:
        is_pos = False

    is_neg = (score <= q10_score) and (dw <= q25_dw)
    if is_neg and is_contact == 1 and dist <= juxtacrine_distance:
        is_neg = False

    if is_pos:
        pseudo_labels[i] = 1
    elif is_neg:
        pseudo_labels[i] = 0

edge_meta_df["pseudo_label"] = pseudo_labels

print("Pseudo-label summary:")
print("  positive:", int(np.sum(pseudo_labels == 1)))
print("  negative:", int(np.sum(pseudo_labels == 0)))
print("  ignore  :", int(np.sum(pseudo_labels == -1)))

In [ ]:
edge_meta_df = edge_meta_df.sort_values("edge_idx").reset_index(drop=True)

scores_out = edge_meta_df["sccomm_like_score"].values.astype(np.float32)
labels_out = edge_meta_df["pseudo_label"].values.astype(np.int64)
with gzip.open(args.data_to + args.data_name + '_scores_out2', "wb") as fp:
    pickle.dump(scores_out, fp)
with gzip.open(args.data_to + args.data_name + '_pseudo_labels2', "wb") as fp:
    pickle.dump(labels_out, fp)
with gzip.open(args.data_to + args.data_name + '_adjacency_records2', 'wb') as fp:  
    pickle.dump([row_col, edge_weight, lig_rec, total_num_cell,w1_list, w2_list, w3_list], fp)